# M1-B1 — Exploration de l'historique des réservations

Ce notebook sert aux tâches 1 à 3 : connexion à la base, exploration en SQL, chargement des données.
Crée d'autres notebooks pour la suite (par exemple `02_modelisation.ipynb`).

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # pour pouvoir importer model.train_baseline plus tard
load_dotenv(ROOT / ".env")

engine = create_engine(os.environ["DATABASE_URL"])

## Tâche 1 — La base répond-elle ?

In [2]:
pd.read_sql(text("SELECT COUNT(*) AS nb_reservations FROM reservations"), engine)

,nb_reservations
0,78703


## Tâche 2 — Explorer en SQL

Les calculs se font **dans la requête SQL** ; pandas sert seulement à afficher le résultat.

Pour commencer : les tables disponibles et leurs colonnes.

In [3]:
pd.read_sql(text("SELECT * FROM hotels"), engine)

,hotel_id,code,nom,type,ville
0,1,City Hotel,Escale Atlantique Centre,urbain,Lisbonne
1,2,Resort Hotel,Escale Atlantique Océan,resort,Algarve


In [4]:
pd.read_sql(text("SELECT * FROM reservations LIMIT 5"), engine)

,reservation_id,hotel_id,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
0,1,1,0,6,2015,July,27,1,0,2,...,6,None,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01
1,2,1,1,88,2015,July,27,1,0,4,...,9,None,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01
2,3,1,1,65,2015,July,27,1,0,4,...,9,None,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01
3,4,1,1,92,2015,July,27,1,2,4,...,9,None,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01
4,5,1,1,257,2015,July,27,1,0,2,...,6,None,0,Transient,101.5,0,0,Canceled,2015-06-16,2015-07-01


### Réservations et taux d'annulation par hôtel

In [5]:
query = text("""
    SELECT
        h.hotel_id,
        h.code,
        h.nom AS nom_hotel,
        h.type,
        h.ville,
        COUNT(r.reservation_id) AS nombre_reservations,
        SUM(r.is_canceled) AS nombre_annulations,
        ROUND(
            SUM(r.is_canceled) * 100.0 / COUNT(r.reservation_id),
            2
        ) AS taux_annulation
    FROM reservations r
    INNER JOIN hotels h
        ON r.hotel_id = h.hotel_id
    GROUP BY
        h.hotel_id,
        h.code,
        h.nom,
        h.type,
        h.ville
    ORDER BY taux_annulation DESC
""")
# Exécuter la requête et afficher les résultats
df_hotel = pd.read_sql(query, engine)

df_hotel

,hotel_id,code,nom_hotel,type,ville,nombre_reservations,nombre_annulations,taux_annulation
0,1,City Hotel,Escale Atlantique Centre,urbain,Lisbonne,51822,21411,41.32
1,2,Resort Hotel,Escale Atlantique Océan,resort,Algarve,26881,7068,26.29


### Taux d'annulation selon le mois d'arrivée

In [6]:
# Calculer le nombre de réservations et le taux d'annulation par mois
query = text("""
    SELECT
        arrival_date_month AS mois_arrivee,
        COUNT(reservation_id) AS nombre_reservations,
        SUM(is_canceled) AS nombre_annulations,
        ROUND(
            SUM(is_canceled) * 100.0 / COUNT(reservation_id),
            2
        ) AS taux_annulation
    FROM reservations
    GROUP BY arrival_date_month
    ORDER BY
        CASE arrival_date_month
            WHEN 'January' THEN 1
            WHEN 'February' THEN 2
            WHEN 'March' THEN 3
            WHEN 'April' THEN 4
            WHEN 'May' THEN 5
            WHEN 'June' THEN 6
            WHEN 'July' THEN 7
            WHEN 'August' THEN 8
            WHEN 'September' THEN 9
            WHEN 'October' THEN 10
            WHEN 'November' THEN 11
            WHEN 'December' THEN 12
        END
""")

# Exécuter la requête et afficher les résultats
df_mois = pd.read_sql(query, engine)

df_mois

,mois_arrivee,nombre_reservations,nombre_annulations,taux_annulation
0,January,2248,557,24.78
1,February,3891,1337,34.36
2,March,4824,1477,30.62
3,April,5428,2061,37.97
4,May,5478,1915,34.96
5,June,5292,2096,39.61
6,July,7348,2758,37.53
7,August,8952,3423,38.24
8,September,10508,4116,39.17
9,October,11160,4246,38.05


### Taux d'annulation selon le délai entre réservation et arrivée

In [9]:
# # Taux d'annulation selon le délai entre réservation et arrivée
query = text("""
    SELECT
        CASE
            WHEN lead_time BETWEEN 0 AND 7 THEN '0-7 jours'
            WHEN lead_time BETWEEN 8 AND 30 THEN '8-30 jours'
            WHEN lead_time BETWEEN 31 AND 90 THEN '31-90 jours'
            WHEN lead_time BETWEEN 91 AND 180 THEN '91-180 jours'
            WHEN lead_time BETWEEN 181 AND 365 THEN '181-365 jours'
            ELSE 'Plus de 365 jours'
        END AS tranche_delai,

        COUNT(*) AS nombre_reservations,
        SUM(is_canceled) AS nombre_annulations,

        ROUND(
            SUM(is_canceled) * 100.0 / COUNT(*),
            2
        ) AS taux_annulation

    FROM reservations
    GROUP BY tranche_delai
    ORDER BY MIN(lead_time)
""")

# Exécuter la requête et afficher les résultats
df_delai = pd.read_sql(query, engine)

df_delai


,tranche_delai,nombre_reservations,nombre_annulations,taux_annulation
0,0-7 jours,13925,1391,9.99
1,8-30 jours,12602,3342,26.52
2,31-90 jours,20140,7159,35.55
3,91-180 jours,15748,6837,43.42
4,181-365 jours,14504,8365,57.67
5,Plus de 365 jours,1784,1385,77.63


### Taux d'annulation par segment de clientèle et par type d'acompte

In [8]:
# # Taux d'annulation par segment de clientèle et type d'acompte
query = text("""
    SELECT
        customer_type AS segment_clientele,
        deposit_type AS type_acompte,

        COUNT(reservation_id) AS nombre_reservations,
        SUM(is_canceled) AS nombre_annulations,

        ROUND(
            SUM(is_canceled) * 100.0 / COUNT(reservation_id),
            2
        ) AS taux_annulation

    FROM reservations

    GROUP BY customer_type, deposit_type

    ORDER BY taux_annulation DESC
""")

# Exécuter la requête et afficher les résultats
df_segment = pd.read_sql(query, engine)

df_segment

,segment_clientele,type_acompte,nombre_reservations,nombre_annulations,taux_annulation
0,Contract,Non Refund,544,544,100.00
1,Transient,Non Refund,8796,8796,100.00
2,Transient-Party,Non Refund,1121,1036,92.42
3,Transient,No Deposit,47026,13096,27.85
4,Transient-Party,No Deposit,17860,4311,24.14
5,Contract,No Deposit,2870,641,22.33
6,Group,No Deposit,351,45,12.82
7,Transient-Party,Refundable,127,10,7.87
8,Group,Refundable,8,0,0.00


### Rapport d'étonnement

*Note ici tes observations.*

## Tâche 3 — Charger les données et préparer le découpage

Une seule requête SQL, avec la jointure sur `hotels` pour récupérer la colonne `hotel`
(`'City Hotel'` / `'Resort Hotel'`) attendue par la recette de la v1.

In [11]:
# Chargement des données
import pandas as pd
from sqlalchemy import text

query = text("""
    SELECT
        r.*,
        h.code AS hotel
    FROM reservations AS r
    INNER JOIN hotels AS h
        ON r.hotel_id = h.hotel_id
""")

df = pd.read_sql(query, engine)

# Convertir la date d'arrivée
df["arrival_date"] = pd.to_datetime(df["arrival_date"])

df.head()

,reservation_id,hotel_id,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,...,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,hotel
0,1,1,0,6,2015,July,27,1,0,2,...,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-03,2015-07-01,City Hotel
1,2,1,1,88,2015,July,27,1,0,4,...,NaN,0,Transient,76.5,0,1,Canceled,2015-07-01,2015-07-01,City Hotel
2,3,1,1,65,2015,July,27,1,0,4,...,NaN,0,Transient,68.0,0,1,Canceled,2015-04-30,2015-07-01,City Hotel
3,4,1,1,92,2015,July,27,1,2,4,...,NaN,0,Transient,76.5,0,2,Canceled,2015-06-23,2015-07-01,City Hotel
4,5,1,1,257,2015,July,27,1,0,2,...,NaN,0,Transient,101.5,0,0,Canceled,2015-06-16,2015-07-01,City Hotel


In [12]:
# 2. Découpage temporel
# Date limite : 4 mois avant la dernière date d'arrivée
date_limite = df["arrival_date"].max() - pd.DateOffset(months=4)

# Entraînement sur les données passées
df_train = df[df["arrival_date"] <= date_limite].copy()

# Test sur les données les plus récentes
df_test = df[df["arrival_date"] > date_limite].copy()

print("Date limite :", date_limite)
print("Train :", df_train.shape)
print("Test :", df_test.shape)

Date limite : 2016-08-31 00:00:00
Train : (58792, 35)
Test : (19911, 35)
